In [ ]:
# Colab: GCP setup + buckets + W&B Secret (robust) =====
from google.colab import auth
import os, subprocess, pathlib, getpass

# small shell helpers
def sh(cmd):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=False)

def gval(cmd):
    return subprocess.check_output(cmd, shell=True).decode().strip()

print("Authenticating your Google Account...")
auth.authenticate_user()

# config
PROJECT_ID = "msda-llm-finetuning"
REGION     = "us-central1"

RAW_BUCKET    = "msda-llm-raw-data"
PROC_BUCKET   = "msda-llm-processed-data"
MODELS_BUCKET = "msda-llm-models"
BENCH_BUCKET  = "msda-llm-benchmarks"    # held-out benchmarks (recommended)

print("\nSetting active project/region...")
sh(f"gcloud config set project {PROJECT_ID}")
sh(f"gcloud config set compute/region {REGION}")

print("\nEnabling core APIs (idempotent)...")
sh(
  "gcloud services enable "
  "aiplatform.googleapis.com storage.googleapis.com compute.googleapis.com "
  "artifactregistry.googleapis.com run.googleapis.com secretmanager.googleapis.com "
  f"--project {PROJECT_ID}"
)

print("\nCreating Google Cloud Storage buckets (idempotent, UBLA on)...")
sh(f"gsutil mb -b on -p {PROJECT_ID} -l {REGION} gs://{RAW_BUCKET}    || true")
sh(f"gsutil mb -b on -p {PROJECT_ID} -l {REGION} gs://{PROC_BUCKET}   || true")
sh(f"gsutil mb -b on -p {PROJECT_ID} -l {REGION} gs://{MODELS_BUCKET} || true")
sh(f"gsutil mb -b on -p {PROJECT_ID} -l {REGION} gs://{BENCH_BUCKET}  || true")

print("\nUBLA status (should be Enabled):")
sh(f"gsutil uniformbucketlevelaccess get gs://{RAW_BUCKET}")
sh(f"gsutil uniformbucketlevelaccess get gs://{PROC_BUCKET}")
sh(f"gsutil uniformbucketlevelaccess get gs://{MODELS_BUCKET}")
sh(f"gsutil uniformbucketlevelaccess get gs://{BENCH_BUCKET}")

print("\nBuckets in project:")
sh(f"gsutil ls -p {PROJECT_ID}")

# W&B Secret Manager setup
print("\nSetting up W&B secret for Vertex/Cloud Run runs...")
wandb_key = os.environ.get("WANDB_API_KEY", "696811cb6ffd112d3efc03edb28b938b9f0fd21a").strip()
if not wandb_key:
    # Secure prompt (hidden). Press Enter to skip if you truly don't want a secret.
    try:
        wandb_key = getpass.getpass("Enter your W&B API key (hidden). Leave blank to skip: ").strip()
    except Exception:
        wandb_key = ""

if wandb_key:
    # Create or update secret safely (no key echoed to shell)
    pathlib.Path("/tmp/wandb_key.txt").write_text(wandb_key)
    sh(f"gcloud secrets create wandb_api_key --project {PROJECT_ID} "
       f"--replication-policy=automatic --data-file=/tmp/wandb_key.txt || "
       f"gcloud secrets versions add wandb_api_key --project {PROJECT_ID} --data-file=/tmp/wandb_key.txt")

    # Grant access to identities that usually read the secret
    PROJECT_NUMBER = gval(f'gcloud projects describe {PROJECT_ID} --format="value(projectNumber)"')
    DEFAULT_COMPUTE_SA   = f"{PROJECT_NUMBER}-compute@developer.gserviceaccount.com"
    VERTEX_SERVICE_AGENT = f"service-{PROJECT_NUMBER}@gcp-sa-aiplatform.iam.gserviceaccount.com"

    sh(f"gcloud secrets add-iam-policy-binding wandb_api_key --project {PROJECT_ID} "
       f"--member=serviceAccount:{DEFAULT_COMPUTE_SA} --role=roles/secretmanager.secretAccessor || true")
    sh(f"gcloud secrets add-iam-policy-binding wandb_api_key --project {PROJECT_ID} "
       f"--member=serviceAccount:{VERTEX_SERVICE_AGENT} --role=roles/secretmanager.secretAccessor || true")

    # If you use a custom training service account, set this in Colab before running:
    # os.environ['TRAINING_SA_EMAIL'] = "vertex-training-sa@msda-llm-finetuning.iam.gserviceaccount.com"
    TRAINING_SA_EMAIL = os.environ.get("TRAINING_SA_EMAIL", "").strip()
    if TRAINING_SA_EMAIL:
        sh(f"gcloud secrets add-iam-policy-binding wandb_api_key --project {PROJECT_ID} "
           f"--member=serviceAccount:{TRAINING_SA_EMAIL} --role=roles/secretmanager.secretAccessor || true")

    print("\nW&B secret ready.")
else:
    print("No W&B key provided. That's fine for Colab-only runs. "
          "If you later launch Vertex jobs or Cloud Run, re-run after setting "
          "os.environ['WANDB_API_KEY']='<key>' or provide it at the prompt above.")

print("\nDone. You’re ready for Phase 1.")

Authenticating your Google Account...

Setting active project/region...
$ gcloud config set project msda-llm-finetuning
$ gcloud config set compute/region us-central1

Enabling core APIs (idempotent)...
$ gcloud services enable aiplatform.googleapis.com storage.googleapis.com compute.googleapis.com artifactregistry.googleapis.com run.googleapis.com secretmanager.googleapis.com --project msda-llm-finetuning

Creating Google Cloud Storage buckets (idempotent, UBLA on)...
$ gsutil mb -b on -p msda-llm-finetuning -l us-central1 gs://msda-llm-raw-data    || true
$ gsutil mb -b on -p msda-llm-finetuning -l us-central1 gs://msda-llm-processed-data   || true
$ gsutil mb -b on -p msda-llm-finetuning -l us-central1 gs://msda-llm-models || true
$ gsutil mb -b on -p msda-llm-finetuning -l us-central1 gs://msda-llm-benchmarks  || true

UBLA status (should be Enabled):
$ gsutil uniformbucketlevelaccess get gs://msda-llm-raw-data
$ gsutil uniformbucketlevelaccess get gs://msda-llm-processed-data
$ gs